# F98 – SQUID: Data analysis

All paths are relative to the repository root. Measurement data are read from `F98/Emilian_Fritz`, figures are saved to `plots/`.

In [ ]:
# Imports
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from IPython.display import display, Math

# Relative paths
DATA_DIR = Path("F98") / "Emilian_Fritz"  # measurement data
PLOT_DIR = Path("plots")  # saved figures
PLOT_DIR.mkdir(exist_ok=True)

# Constants
K_BOLTZMANN = 1.380649e-23  # J/K
V_GAIN_521 = 2000  # voltage amplifier gain in 5.2.1

# 5.2.1 – Characteristics of the single SQUID

## I-V characteristic at integer flux

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_int = np.loadtxt(
    DATA_DIR / "5_2_1" / "30A_voltage_char_int" / "30A_voltage_char_int_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_int = -10
t_max_int = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 30 µA
I_pp_int = 30  # µA
U_pp_int = data_int[:, 1].max() - data_int[:, 1].min()  # V
cal_int = I_pp_int / U_pp_int  # µA/V

# Extract measurement data
t_int = data_int[:, 0]  # ms
I_int = data_int[:, 1] * cal_int  # µA
V_int = data_int[:, 2]  # mV

# Select data within the time interval
mask_int = (t_int >= t_min_int) & (t_int <= t_max_int)
I_sel_int = I_int[mask_int]
V_sel_int = V_int[mask_int]

# Fit boundaries (µA)
I1_int = 5.5
I2_int = 14

# Plot: I-V characteristic
plt.figure(figsize=(8, 5))
plt.plot(I_sel_int, V_sel_int, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_int, linestyle="--", linewidth=0.8, color="red", label="Fit boundaries")
plt.axvline(I2_int, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("I-V characteristic (integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot1.png", format="png")
plt.show()

In [ ]:
# Select data between the fit boundaries
mask_fit_int = mask_int & (I_int >= I1_int) & (I_int <= I2_int)
I_fit_int = I_int[mask_fit_int]
V_fit_int = V_int[mask_fit_int]

# Linear fit including covariance matrix
(m_int, b_int), cov_int = np.polyfit(I_fit_int, V_fit_int, 1, cov=True)
dm_int = np.sqrt(cov_int[0, 0])
db_int = np.sqrt(cov_int[1, 1])

# Plot: linear fit
I_line_int = np.linspace(I_fit_int.min(), I_fit_int.max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(I_fit_int, V_fit_int, s=10, marker=".", color="blue", label="Measured data")
plt.plot(I_line_int, m_int * I_line_int + b_int, color="red", label="Linear fit")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Linear fit of the resistive branch (integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot2.png", format="png")
plt.show()

# Normal-state resistance: mV/µA = kΩ -> ×1000 for Ω, divided by the amplifier gain
R_int = m_int * 1000 / V_GAIN_521  # Ω
dR_int = dm_int * 1000 / V_GAIN_521  # Ω

print(f"Slope:                   m   = ({m_int:.4f} ± {dm_int:.4f}) mV/µA")
print(f"Intercept:               b   = ({b_int:.3f} ± {db_int:.3f}) mV")
print(f"Normal-state resistance: R_N = ({R_int:.4f} ± {dR_int:.4f}) Ω")

## I-V characteristic at half-integer flux

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_half = np.loadtxt(
    DATA_DIR / "5_2_1" / "30A_voltage_char_half" / "30A_voltage_char_half_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_half = -10
t_max_half = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 30 µA
I_pp_half = 30  # µA
U_pp_half = data_half[:, 1].max() - data_half[:, 1].min()  # V
cal_half = I_pp_half / U_pp_half  # µA/V

# Extract measurement data
t_half = data_half[:, 0]  # ms
I_half = data_half[:, 1] * cal_half  # µA
V_half = data_half[:, 2]  # mV

# Select data within the time interval
mask_half = (t_half >= t_min_half) & (t_half <= t_max_half)
I_sel_half = I_half[mask_half]
V_sel_half = V_half[mask_half]

# Fit boundaries (µA)
I1_half = -0.5
I2_half = 10

# Plot: I-V characteristic
plt.figure(figsize=(8, 5))
plt.plot(I_sel_half, V_sel_half, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_half, linestyle="--", linewidth=0.8, color="red", label="Fit boundaries")
plt.axvline(I2_half, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("I-V characteristic (half-integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot3.png", format="png")
plt.show()

In [ ]:
# Select data between the fit boundaries
mask_fit_half = mask_half & (I_half >= I1_half) & (I_half <= I2_half)
I_fit_half = I_half[mask_fit_half]
V_fit_half = V_half[mask_fit_half]

# Linear fit including covariance matrix
(m_half, b_half), cov_half = np.polyfit(I_fit_half, V_fit_half, 1, cov=True)
dm_half = np.sqrt(cov_half[0, 0])
db_half = np.sqrt(cov_half[1, 1])

# Plot: linear fit
I_line_half = np.linspace(I_fit_half.min(), I_fit_half.max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(I_fit_half, V_fit_half, s=10, marker=".", color="blue", label="Measured data")
plt.plot(I_line_half, m_half * I_line_half + b_half, color="red", label="Linear fit")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Linear fit of the resistive branch (half-integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot4.png", format="png")
plt.show()

# Normal-state resistance: mV/µA = kΩ -> ×1000 for Ω, divided by the amplifier gain
R_half = m_half * 1000 / V_GAIN_521  # Ω
dR_half = dm_half * 1000 / V_GAIN_521  # Ω

print(f"Slope:                   m   = ({m_half:.4f} ± {dm_half:.4f}) mV/µA")
print(f"Intercept:               b   = ({b_half:.3f} ± {db_half:.3f}) mV")
print(f"Normal-state resistance: R_N = ({R_half:.4f} ± {dR_half:.4f}) Ω")

## Resistance of the SQUID $R_{\mathrm{SQUID}}$ and of a single Josephson junction $R_{\mathrm{J}}$

In [ ]:
# Mean of integer and half-integer flux
R_SQUID = (R_int + R_half) / 2  # Ω
dR_SQUID = np.sqrt(dR_int**2 + dR_half**2) / 2  # Ω

# Two junctions in parallel: R_J = 2 R_SQUID
R_J = 2 * R_SQUID  # Ω
dR_J = 2 * dR_SQUID  # Ω

print(f"SQUID resistance:              R_SQUID = ({R_SQUID:.3f} ± {dR_SQUID:.3f}) Ω")
print(f"Josephson junction resistance: R_J     = ({R_J:.3f} ± {dR_J:.3f}) Ω")

## Critical current $I_c$

In [ ]:
# Wider time interval of the integer-flux measurement (ms)
t_min_Ic = -30
t_max_Ic = 10

# Select data within the time interval
mask_Ic = (t_int >= t_min_Ic) & (t_int <= t_max_Ic)
t_sel_Ic = t_int[mask_Ic]
V_sel_Ic = V_int[mask_Ic]

# Times of the transitions into the voltage state (ms)
t1_Ic = -17
t2_Ic = -3

# Plot: voltage vs. time
plt.figure(figsize=(8, 5))
plt.plot(t_sel_Ic, V_sel_Ic, marker=".", markersize=0.1, color="blue")
plt.axvline(t1_Ic, linestyle="--", linewidth=0.8, color="red", label="Transition points")
plt.axvline(t2_Ic, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Time t (ms)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage vs. time (integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot5.png", format="png")
plt.show()

In [ ]:
# Current ramp from -I_pp/2 to +I_pp/2 between these times (ms)
t_ramp_start = -10
t_ramp_end = 10

# Reading uncertainty of the transition times (ms)
dt_Ic = 1.0

# Currents at the transition points (µA)
I1_Ic = -I_pp_int / 2 + I_pp_int / (t_ramp_end - t_ramp_start) * (t1_Ic - t_ramp_start)
I2_Ic = -I_pp_int / 2 + I_pp_int / (t_ramp_end - t_ramp_start) * (t2_Ic - t_ramp_start)
dI_Ic = I_pp_int / (t_ramp_end - t_ramp_start) * dt_Ic

# Critical current
I_c = (abs(I1_Ic) + abs(I2_Ic)) / 2  # µA
dI_c = np.sqrt(2) * dI_Ic / 2  # µA

print(f"I_1 = ({I1_Ic:.2f} ± {dI_Ic:.2f}) µA")
print(f"I_2 = ({I2_Ic:.2f} ± {dI_Ic:.2f}) µA")
print(f"I_c = ({I_c:.2f} ± {dI_c:.2f}) µA")

## Inverse mutual inductance of the feedback coil $M_{\Phi_B}^{-1}$

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_B = np.loadtxt(
    DATA_DIR / "5_2_1" / "V_phi_peak_peak_max_center" / "V_phi_peak_peak_max_center_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_B = -10
t_max_B = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 180.06 µA
I_pp_B = 180.06  # µA
U_pp_B = data_B[:, 1].max() - data_B[:, 1].min()  # V
cal_B = I_pp_B / U_pp_B  # µA/V

# Extract measurement data
t_B = data_B[:, 0]  # ms
I_B = data_B[:, 1] * cal_B  # µA
V_B = data_B[:, 2]  # mV

# Select data within the time interval
mask_B = (t_B >= t_min_B) & (t_B <= t_max_B)
I_sel_B = I_B[mask_B]
V_sel_B = V_B[mask_B]

# Period boundaries (µA)
I1_B = -60
I2_B = 56

# Plot: V-I characteristic of the feedback coil
plt.figure(figsize=(8, 5))
plt.plot(I_sel_B, V_sel_B, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_B, linestyle="--", linewidth=0.8, color="red", label="Period boundaries")
plt.axvline(I2_B, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Feedback coil current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (feedback coil)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot6.png", format="png")
plt.show()

In [ ]:
# Reading uncertainty of the period boundaries (µA)
dI_B = 2.0

# Number of flux quanta between the period boundaries
N_B = 3

# Inverse mutual inductance
M_inv_B = (I2_B - I1_B) / N_B  # µA/Φ0
dM_inv_B = np.sqrt(2) * dI_B / N_B  # µA/Φ0

display(Math(
    rf"M_{{\Phi_B}}^{{-1}} = ({M_inv_B:.3f} \pm {dM_inv_B:.3f})"
    r"\,\mu\mathrm{A}/\Phi_0"
))

In [ ]:
# Convert current to flux; the offset is chosen, the absolute flux origin is not calibrated
phi_sel_B = I_sel_B / M_inv_B + 0.05  # Φ0

# Plot: V-Φ characteristic
plt.figure(figsize=(8, 5))
plt.plot(phi_sel_B, V_sel_B, marker=".", markersize=0.1, color="blue")
plt.xlabel(r"Flux $\Phi / \Phi_0$")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (feedback coil)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot7.png", format="png")
plt.show()

## Inverse mutual inductance of the input coil $M_{\mathrm{IN}}^{-1}$

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_IN = np.loadtxt(
    DATA_DIR / "5_2_1" / "V_phi_x_peak_peak_max_center" / "V_phi_x_peak_peak_max_center_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_IN = -10
t_max_IN = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 100 µA
I_pp_IN = 100  # µA
U_pp_IN = data_IN[:, 1].max() - data_IN[:, 1].min()  # V
cal_IN = I_pp_IN / U_pp_IN  # µA/V

# Extract measurement data
t_IN = data_IN[:, 0]  # ms
I_IN = data_IN[:, 1] * cal_IN  # µA
V_IN = data_IN[:, 2]  # mV

# Select data within the time interval
mask_IN = (t_IN >= t_min_IN) & (t_IN <= t_max_IN)
I_sel_IN = I_IN[mask_IN]
V_sel_IN = V_IN[mask_IN]

# Period boundaries (µA)
I1_IN = -40.5
I2_IN = 39.5

# Plot: V-I characteristic of the input coil
plt.figure(figsize=(8, 5))
plt.plot(I_sel_IN, V_sel_IN, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_IN, linestyle="--", linewidth=0.8, color="red", label="Period boundaries")
plt.axvline(I2_IN, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Input coil current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (input coil)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot8.png", format="png")
plt.show()

In [ ]:
# Reading uncertainty of the period boundaries (µA)
dI_IN = 2.0

# Number of flux quanta between the period boundaries
N_IN = 9

# Inverse mutual inductance
M_inv_IN = (I2_IN - I1_IN) / N_IN  # µA/Φ0
dM_inv_IN = np.sqrt(2) * dI_IN / N_IN  # µA/Φ0

display(Math(
    rf"M_{{\mathrm{{IN}}}}^{{-1}} = ({M_inv_IN:.3f} \pm {dM_inv_IN:.3f})"
    r"\,\mu\mathrm{A}/\Phi_0"
))

In [ ]:
# Convert current to flux; the offset is chosen, the absolute flux origin is not calibrated
phi_sel_IN = I_sel_IN / M_inv_IN + 0.05  # Φ0

# Plot: V-Φ characteristic
plt.figure(figsize=(8, 5))
plt.plot(phi_sel_IN, V_sel_IN, marker=".", markersize=0.1, color="blue")
plt.xlabel(r"Flux $\Phi / \Phi_0$")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (input coil)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot9.png", format="png")
plt.show()

# 5.2.2 – Single-stage flux-locked loop (FLL)

## FLL characteristic

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: output voltage in V)
data_FLL = np.loadtxt(
    DATA_DIR / "5_2_2" / "single_FLL" / "single_FLL_2.txt",
    skiprows=3,
)

# Time interval for the linear fit (ms)
t_min_FLL = -10
t_max_FLL = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 20 µA
I_pp_FLL = 20  # µA
U_pp_FLL = data_FLL[:, 1].max() - data_FLL[:, 1].min()  # V
cal_FLL = I_pp_FLL / U_pp_FLL  # µA/V

# Extract measurement data
t_FLL = data_FLL[:, 0]  # ms
I_FLL = data_FLL[:, 1] * cal_FLL  # µA
V_FLL = data_FLL[:, 2]  # V

# Select data within the time interval
mask_FLL = (t_FLL >= t_min_FLL) & (t_FLL <= t_max_FLL)
I_fit_FLL = I_FLL[mask_FLL]
V_fit_FLL = V_FLL[mask_FLL]

# Linear fit including covariance matrix
(m_FLL, b_FLL), cov_FLL = np.polyfit(I_fit_FLL, V_fit_FLL, 1, cov=True)
dm_FLL = np.sqrt(cov_FLL[0, 0])

print(f"Slope: m = ({m_FLL:.4f} ± {dm_FLL:.4f}) V/µA")

# Plot: FLL characteristic
plt.figure(figsize=(8, 5))
plt.plot(I_FLL, V_FLL, marker=".", linestyle="None", markersize=1, color="blue", label="Measured data")
plt.plot(I_fit_FLL, m_FLL * I_fit_FLL + b_FLL, linewidth=2, color="red", label="Linear fit")
plt.xlabel("Input current I (µA)")
plt.ylabel("Output voltage V (V)")
plt.title("FLL characteristic")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot10.png", format="png")
plt.show()

## FLL amplification

In [ ]:
# Feedback resistance (kΩ)
R_F_522 = 100

# Theoretical amplification: kΩ · µA/µA = mV/µA -> ×1e-3 for V/µA
A_theo = R_F_522 * (M_inv_B / M_inv_IN) * 1e-3  # V/µA
dA_theo = A_theo * np.sqrt((dM_inv_IN / M_inv_IN)**2 + (dM_inv_B / M_inv_B)**2)  # V/µA

# Experimental amplification
A_exp = np.abs(m_FLL)  # V/µA
dA_exp = dm_FLL  # V/µA

# Deviation in standard deviations
n_sigma_A = np.abs(A_theo - A_exp) / np.sqrt(dA_exp**2 + dA_theo**2)

print(f"Theoretical amplification:  A_theo = ({A_theo:.3f} ± {dA_theo:.3f}) V/µA")
print(f"Experimental amplification: A_exp  = ({A_exp:.3f} ± {dA_exp:.3f}) V/µA")
print(f"Deviation:                  {n_sigma_A:.2f} σ")

# 5.2.3 – SQUID noise

## Noise spectrum

In [ ]:
# Load measurement data (frequency in MHz, channel B: voltage level in dBV)
data_noise = np.loadtxt(
    DATA_DIR / "5_2_3" / "30_kOhm_1_5_ghz" / "30_kOhm_1_5_ghz_2.txt",
    skiprows=3,
)

# Extract measurement data
f_noise = data_noise[:, 0]  # MHz
L_noise = data_noise[:, 1]  # dBV

# Plot: noise spectrum
plt.figure(figsize=(8, 5))
plt.plot(f_noise, L_noise, marker=".", linestyle="None", markersize=1, color="blue", label="Measured data")
plt.xscale("log")
plt.xlabel("Frequency f (MHz)")
plt.ylabel("Voltage level (dBV)")
plt.title("Noise spectrum")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot11.png", format="png")
plt.show()

# 5.3.1 – Two-stage setup

## Voltage-flux characteristic ($\Phi_B$)

In [ ]:
# Load measurement data (time in ms, channel A: flux ramp in V, channel C: voltage in mV)
data_531 = np.loadtxt(
    DATA_DIR / "5_3_1" / "Phi_V_char_Phi_B" / "Phi_V_char_Phi_B_2.txt",
    skiprows=3,
)

# Extract measurement data
t_531 = data_531[:, 0]  # ms
U_ramp_531 = data_531[:, 1]  # V
V_531 = data_531[:, 2]  # mV

# Plot: voltage-flux characteristic
plt.figure(figsize=(8, 5))
plt.plot(U_ramp_531, V_531, linewidth=1.5, color="blue")
plt.xlabel("Flux ramp signal (V)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (two-stage setup)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot12.png", format="png")
plt.show()

# 5.3.2 – Noise thermometry with the two-stage setup

## Transimpedance gain

In [ ]:
# The gain A_exp from 5.2.2 was measured with R_F = 100 kΩ.
# It scales linearly with the feedback resistance; this measurement used R_F = 30 kΩ.
R_F_532 = 30  # kΩ
gain_factor = R_F_532 / R_F_522

A_532 = gain_factor * A_exp  # V/µA
dA_532 = gain_factor * dA_exp  # V/µA

print(f"Transimpedance gain: A = ({A_532:.4f} ± {dA_532:.4f}) V/µA")

## Current noise spectrum and fit

In [ ]:
# Load measurement data (frequency in MHz, channel C: voltage level in dBV)
data_532 = np.loadtxt(
    DATA_DIR / "5_3_2" / "two_stage_30kO_4GHz" / "two_stage_30kO_4GHz_1.txt",
    skiprows=3,
)

# Extract measurement data
f_532 = data_532[:, 0] * 1e6  # Hz
L_532 = data_532[:, 1]  # dBV

# Voltage noise density: sqrt(S_V) = 10^(L/20) / sqrt(Δf)
delta_f_532 = np.abs(f_532[2] - f_532[1])  # Hz, frequency resolution
sqrtS_V_532 = 10 ** (L_532 / 20) / np.sqrt(delta_f_532)  # V/√Hz

# Current noise density: V/µA -> V/A with ×1e6
sqrtS_I_532 = sqrtS_V_532 / (A_532 * 1e6)  # A/√Hz

# Input inductance (H)
L_IN = 2.9e-9

# Model: thermal current noise of a resistor R, low-pass filtered by L_IN
def sqrt_S_I(f, T, R):
    f_c = R / (2 * np.pi * L_IN)  # Hz
    return np.sqrt(4 * K_BOLTZMANN * T / R / (1 + (f / f_c)**2))  # A/√Hz

# Fit range (indices)
i_min_532 = 50
i_max_532 = 3500
f_fit_532 = f_532[i_min_532:i_max_532]
sqrtS_I_fit_532 = sqrtS_I_532[i_min_532:i_max_532]

# The fit runs in rescaled units (√S_I in pA/√Hz, R in mΩ), because the raw values are
# very small and the fit is numerically ill-conditioned otherwise
def sqrt_S_I_scaled(f, T, R_mOhm):
    return sqrt_S_I(f, T, R_mOhm * 1e-3) * 1e12  # pA/√Hz

popt_532, pcov_532 = curve_fit(
    sqrt_S_I_scaled, f_fit_532, sqrtS_I_fit_532 * 1e12,
    p0=[1.0, 10.0],  # T = 1 K, R = 10 mΩ
    bounds=([0, 0], [np.inf, np.inf]),  # T > 0, R > 0
)
T_532, R_532 = popt_532  # K, mΩ
dT_532, dR_532 = np.sqrt(np.diag(pcov_532))  # K, mΩ
f_c_532 = R_532 * 1e-3 / (2 * np.pi * L_IN)  # Hz

print(f"Temperature:      T   = ({T_532:.3f} ± {dT_532:.3f}) K")
print(f"Resistance:       R   = ({R_532:.2f} ± {dR_532:.2f}) mΩ")
print(f"Cutoff frequency: f_c = {f_c_532 / 1e3:.0f} kHz")

# Plot: current noise spectrum with fit
plt.figure(figsize=(12, 7))
plt.loglog(f_532 / 1e6, sqrtS_I_532, color="dodgerblue", label="Measured data")
plt.loglog(f_fit_532 / 1e6, sqrt_S_I(f_fit_532, T_532, R_532 * 1e-3), linewidth=3, color="tomato", label="Fit")
plt.axvline(f_532[i_min_532] / 1e6, linestyle="--", color="gray", label="Fit interval")
plt.axvline(f_532[i_max_532 - 1] / 1e6, linestyle="--", color="gray")
plt.text(
    0.02, 0.05,
    rf"$T = ({T_532:.3f} \pm {dT_532:.3f})$ K" "\n" rf"$R = ({R_532:.2f} \pm {dR_532:.2f})$ m$\Omega$",
    transform=plt.gca().transAxes, fontsize=12,
)
plt.xlim(1e-3, 5)
plt.xlabel("Frequency f (MHz)", size=14)
plt.ylabel(r"Current noise density $\sqrt{S_I}$ (A/$\sqrt{\mathrm{Hz}}$)", size=14)
plt.title(r"Current noise spectrum (two-stage setup, $R_F$ = 30 k$\Omega$)", size=16)
plt.legend(loc="best")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot13.png", format="png")
plt.show()